In [1]:
import ollama
from openai import OpenAI
from dotenv import load_dotenv
import os
import pandas as pd


In [2]:
load_dotenv()
OPENAI_KEY = os.environ.get("OPENAI_API_KEY")

In [3]:
dir_path = "../data/base_models/instruct/" 
datasets = ["mistral_7b"]# ,"openbiollm_8b", "gemma2_9b", "medGemma_4b", "medGemma_27b"]

In [4]:
def create_mcq_text(mcq_dict):
    return (
        f"Question: {mcq_dict['question']}\n"
        f"a) {mcq_dict['option_a']}\n"
        f"b) {mcq_dict['option_b']}\n"
        f"c) {mcq_dict['option_c']}\n"
        f"d) {mcq_dict['option_d']}"
    )

In [5]:
def llama_answer_qcm(mcq_text,system_prompt):
    user_prompt = f"""Répond STRICTEMENT à ce QCM :
        {mcq_text}
        CONTRAINTE ABSOLUE :
        - Ta sortie doit être UNIQUEMENT la lettre de la bonne réponse (A, B, C, D, etc.).
        - AUCUN autre texte, aucune explication, aucun point, aucun saut de ligne, aucun espace.
        - Ne préfixe pas la réponse, n’ajoute rien avant ou après.
        - Répond par une seule lettre et rien d’autre.

        FORMAT DE SORTIE OBLIGATOIRE :
        <lettre>
    """
    response = ollama.generate(
        model="llama3.1:70b",
        prompt=user_prompt,
        system=system_prompt)

    return response["response"][0]

In [6]:
def call_openai_api(client, system_prompt, mcq_text, temp=0.5, max_completion_tokens=1):
    user_prompt = f"""Répond STRICTEMENT à ce QCM :
        {mcq_text}
        CONTRAINTE ABSOLUE :
        - Ta sortie doit être UNIQUEMENT la lettre de la bonne réponse (A, B, C, D, etc.).
        - AUCUN autre texte, aucune explication, aucun point, aucun saut de ligne, aucun espace.
        - Ne préfixe pas la réponse, n’ajoute rien avant ou après.
        - Répond par une seule lettre et rien d’autre.

        FORMAT DE SORTIE OBLIGATOIRE :
        <lettre>
         """
    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            temperature=temp,
            max_tokens=max_completion_tokens,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error occurred: {e}")
        return None

In [7]:
system_prompt = "Tu es un expert dans le domaine médical"
client = OpenAI(api_key=OPENAI_KEY)

# How many output are incorrect

In [8]:
print("Number of incorrect format for the correction option")
for dataset in datasets:
    df = pd.read_csv(dir_path + dataset + ".csv")
    initial_len = len(df)
    incorrect_output = 0
    for idx, row in df.iterrows():
        correct_option = row["correct_option"]
        if not(isinstance(correct_option, str)):
            incorrect_output +=1
            continue 
        if correct_option not in "abcdABCD":
            incorrect_output += 1
    print(f"{dataset}: {round((incorrect_output/initial_len)*100,2)}%")

Number of incorrect format for the correction option
mistral_7b: 0.13%


# Correctness

In [9]:
for dataset in datasets:
    df = pd.read_csv(dir_path + dataset + ".csv")

    initial_len = len(df)
    indices_to_drop = []
    for idx, row in df.iterrows():
        mcq_text = create_mcq_text(row)
        correct_option = row["correct_option"]
        correct_reeval = call_openai_api(client=client,system_prompt=system_prompt,mcq_text=mcq_text)
        if  not(isinstance(correct_option, str)):
            continue 
        if correct_option.lower() != correct_reeval.lower():
            indices_to_drop.append(idx)

    df = df.drop(indices_to_drop).reset_index(drop=True)
    df.to_csv("../data/correct_dataset/instruct/"+ dataset + ".csv")
    print(f"Number of correct MCQs for {dataset} {len(df)} / {initial_len}")


Number of correct MCQs for mistral_7b 2102 / 3184
